# Confidence interval of counted data 
## Introduction


## Poisson distribution
### Poisson distribution with NumPy


In [ ]:
import numpy as np

np.random.seed(111) # for reproducibility

# Set the parameters for the Poisson distribution
λ1 = 1.6  # i.e., mean radiation counts/min
λ2 = 7.5
size = 10000

# Generate random variates
data1 = np.random.poisson(lam=λ1, size=size)
data2 = np.random.poisson(λ2, size)

print(
    f"First 15 values of the random variate with {λ1=}:\n",
    data1[:15])
print()
print(
    f"First 15 values of the random variate with {λ2=}:\n",
    data2[:15])

In [ ]:

import matplotlib.pyplot as plt

# Plot histograms using Matplotlib
plt.figure(figsize=(3.5, 3))

plt.hist(
    data1, bins=range(min(data1), max(data1)),
    alpha=0.7, label=f'λ={λ1}')
plt.hist(
    data2, bins=range(min(data2), max(data2)),
    alpha=0.7, label=f'λ={λ2}')
plt.title('Poisson (NumPy and Matplotlib)')
plt.xlabel('Value (counts/min)')
plt.ylabel('Frequency')
plt.xticks(np.arange(0, 19, 1), size=8, rotation=60)
plt.margins(x=0, y=0)
plt.legend(loc='upper right');

### Poisson distributions with SciPy


In [ ]:

from scipy.stats import poisson
import seaborn as sns

plt.figure(figsize=(3.5, 3))
sns.histplot(
    poisson(mu=λ1).rvs(size=size, random_state=111),
    stat='probability',
    binwidth=1,
    discrete=True,
    label=f"λ={λ1}",
    color='royalblue')
sns.histplot(
    poisson(mu=λ2).rvs(size=size, random_state=111),
    stat='probability',
    binwidth=1,
    discrete=True,
    label=f"λ={λ2}",
    color='orangered')

plt.title('Poisson (SciPy and Seaborn)')
plt.xlabel('Value (counts/min)')
plt.ylabel('Probability')
plt.xticks(np.arange(0, 18, 1), size=8, rotation=60)
plt.legend(loc='upper right')
sns.despine();

### Lollipop plots


In [ ]:

plt.figure(figsize=(3.5, 3))
# Loop through the lambda values and plot the PMFs
for i, lambda_val in enumerate([λ1, λ2]):
    # Create a Poisson distribution object
    model = poisson(mu=lambda_val)

    # Generate the x values using ppf (ensure range contains all data)
    x = np.arange(
       0,  # The minimum x value is zero for Poisson
       poisson.ppf(0.999, mu=lambda_val) + 1)

    # Calculate the PMF
    pmf = model.pmf(x)

    # Create the lollipop plot (using different colors and markers)
    plt.stem(
        x, pmf,
        markerfmt=f"C{i}s",  # Use different markers for each lambda
        linefmt=f"C{i}--",   # Use different colors for each lambda
        basefmt=" ",
        label=f"λ = {lambda_val}")

# Add labels and title
plt.xticks(x, rotation=60)  # Show all the integer on x axis
plt.xlabel("Number of events (k)")
plt.ylabel("Probability")
plt.title("Lollipop plots of Poisson PMFs")
plt.margins(x=0.03, y=0.03)
plt.legend();

## Percentile intervals
### Distribution of raisins per bagel 


In [ ]:

# Set the parameters for the Poisson distribution
λ_bagel = 10 # average number of raisins in each bagel
size = 10000

# Generate distribution and random variates
bagel_dist = poisson(mu=λ_bagel)
bagel_variates = bagel_dist.rvs(size=size, random_state=111)

# Calculate the confidence interval (empirical)
conf_int = np.percentile(bagel_variates, [2.5, 97.5])

# Plot histogram
plt.figure(figsize=(3.5, 3))
plt.hist(
    bagel_variates,
    bins=range(min(bagel_variates), max(bagel_variates) + 1),
    alpha=0.7,
    color='skyblue',
    edgecolor='black',)
plt.axvline(
    x=conf_int[0],
    color='red', linestyle='dashed', linewidth=2,
    label=f'2.5th percentile={conf_int[0]:n}')
plt.axvline(
    x=conf_int[1],
    color='green', linestyle='dashed', linewidth=2,
    label=f'97.5th percentile={conf_int[1]:n}')
plt.title(f'Distribution of raisins (λ={λ_bagel})')
plt.xlabel('Number of raisins, k')
plt.ylabel('Frequency')
plt.legend()
sns.despine();

### Normal approximation


In [ ]:
#| fig-subcap: 
#|   - "Different Poisson distributions with varying λ"
#|   - "Comparison of the Poisson and normal distributions"
#| layout-ncol: 2

# Different Poisson distributions with varying λ
lambda_values = [1, 3, 10, 25, 50]

# Use a context manager to change color palette only in this figure
with sns.color_palette("magma_r"):

    # Generate x values (adapted based on the largest lambda, adding few SD)
    max_x = max(lambda_values) + 4 * np.sqrt(max(lambda_values))
    x = np.arange(0, int(max_x) + 1)

    # Create the plot
    plt.figure(figsize=(3.5, 3))
    for lambda_val in lambda_values:
        # Calculate Poisson PMF
        poisson_pmf = poisson(lambda_val).pmf(x)

        # Plot the Poisson distribution (as a bar plot since it's discrete)
        plt.plot(x, poisson_pmf, label=f"λ={lambda_val}")

    # Add labels and title
    plt.xlabel('k (number of events)')
    plt.ylabel('Probability')
    plt.margins(x=0, y=0)
    plt.yticks([])
    plt.title("Poisson with varying λ")
    plt.legend()
    plt.show()  # Required for building multi-plot figures in Quarto

    # ---
    # Comparison of the Poisson and normal distributions

    from scipy.stats import norm

    # Set up parameters
    lambda_max = max(lambda_values)

    # Calculate PMF and PDF values
    poisson_pmf = poisson(lambda_max).pmf(x)
    σ = lambda_max**.5
    norm_pdf = norm(loc=lambda_max, scale=σ).pdf(x)

    # Plot
    plt.figure(figsize=(3.5, 3))
    plt.plot(
        x, poisson_pmf,
        lw=3, ls='--', color='blue',
        label=f"Poisson (λ={lambda_max})")

    plt.plot(x, norm_pdf, lw=3, label=f"Normal (μ={lambda_max}, σ={σ:.2f})")
    plt.title('Poisson and normal distributions')
    plt.xlabel('k (Poisson) or z (normal)')
    plt.ylabel('Probability density')
    plt.margins(x=0, y=0)
    plt.yticks([])
    plt.legend()
    plt.show()

### Prediction interval


In [ ]:
# Calculate the approximate 95% confidence interval for a single observation
pi_normal = np.array(
    (λ_bagel - 1.96 * np.sqrt(λ_bagel), λ_bagel + 1.96 * np.sqrt(λ_bagel)))

print(f"Average number of raisins per bagel λ={λ_bagel}")
print(
    r"95% prediction interval (normal approximation and true λ):",
    pi_normal.astype(int))

In [ ]:
# Calculate the 95% prediction interval using the PPF
pi_ppf = np.array((bagel_dist.ppf(0.025), bagel_dist.ppf(0.975)))

# Print the confidence interval
print(r"95% prediction interval (PPF):", pi_ppf.astype(int))

In [ ]:
# Calculate the 95% prediction interval using the interval method
pi_scipy = bagel_dist.interval(0.95)

print(
    r"95% prediction interval (equal areas around the median; SciPy):",
    pi_scipy)

### Confidence interval of λ


In [ ]:
# Calculate the empirical sample mean and variance
bagel_mean = bagel_variates.mean()
bagel_se = np.sqrt(bagel_mean / size)  # Standard error; size=10000

# Calculate the 95% confidence interval for λ
ci_normal = np.array(
    (bagel_mean - 1.96 * bagel_se, bagel_mean + 1.96 * bagel_se))

print(f"95% CI for λ (normal approximation; {size=}):", ci_normal.round(3))

In [ ]:
size_25 = 25
bagel_variates_25 = bagel_dist.rvs(size=size_25, random_state=111)

# Calculate the empirical sample mean and variance for sample of size 25
bagel_mean_25 = np.mean(bagel_variates_25) #Same as bagel_variates_25.mean()
bagel_se_25 = np.sqrt(bagel_mean_25 / 25)

# Calculate the 95% confidence interval for λ
ci_normal_25 = np.array((
    bagel_mean_25 - 1.96 * bagel_se_25,
    bagel_mean_25 + 1.96 * bagel_se_25))

print(
    f"95% CI for λ (normal approximation; size={size_25}):",
    ci_normal_25.round(3))

### Confidence intervals with statsmodels


In [ ]:
import statsmodels.stats.rates as smr

# Calculate 95% confidence interval using the 'score' method
# Note that confidence level = 1 - α (always refer to documentation ;-)
ci_score = smr.confint_poisson(λ_bagel, 1, alpha=.05, method='score')

print(f"95% confidence interval (score method; statsmodels): \
({ci_score[0]:.2f}, {ci_score[1]:.2f})")

## Modeling high and low event rates
### Example of radioactive count


In [ ]:
# Set the parameters for the Poisson distribution
lambda_radio = 120
# We keep size=10000

# Generate random variates
radio_dist = poisson(mu=lambda_radio)
radio_variates = radio_dist.rvs(size=size, random_state=111)

# Calculate the empirical sample mean and variance
radio_mean = np.mean(radio_variates)
radio_var = np.var(radio_variates, ddof=1)

# Calculate the empirical 95% percentile interval
radio_empirical_pi = np.percentile(radio_variates, [2.5, 97.5])

print(f"Assumed λ (rate parameter): {lambda_radio}")
print(f"Sample mean: {radio_mean:.2f}")
print(f"Sample variance (unbiased): {radio_var:.2f}")
print(r"95% empirical percentile interval:", radio_empirical_pi.round())

# 95% PI using the normal distribution and true lambda
pi_normal = np.array((
    lambda_radio - 1.96 * np.sqrt(lambda_radio),
    lambda_radio + 1.96 * np.sqrt(lambda_radio)))

print(
    r"95% theoretical prediction interval (normal approximation with λ):",
    pi_normal.round(2),)

# Calculate the standard error of the mean
# Use sample mean as estimate of lambda
std_error = np.sqrt(radio_mean / size)

# Calculate the 95% confidence interval for λ
ci_high_rate = np.array(
    (radio_mean - 1.96 * std_error, radio_mean + 1.96 * std_error))

print(
    r"95% CI for λ (normal approximation and sample mean):",
    ci_high_rate.round(3))

In [ ]:
mean_high_rate, variance_high_rate = radio_dist.stats()
std_high_rate = radio_dist.std()
pi_ppf_high_rate = np.array((radio_dist.ppf(0.025),radio_dist.ppf(0.975)))

print(f"Distribution mean: {mean_high_rate:.2f}")
print(f"Distribution variance: {variance_high_rate:.2f}")
print(f"Distribution standard deviation: {std_high_rate:.2f}")
print(r"95% prediction interval (PPF):", pi_ppf_high_rate.astype(int))
print(
    r"95% prediction interval (equal areas around median):",
    radio_dist.interval(0.95))

### The impact of longer time intervals


In [ ]:
# Set the parameters for the Poisson distributions
lambda_lo = 700   # Events per minute
lambda_hi = 7000  # Events per 10 minutes
#size = 10000

# Generate distributions
dist_lambda_lo = poisson(lambda_lo)
dist_lambda_hi = poisson(lambda_hi)

# Generate random variates
variates_lambda_lo = dist_lambda_lo.rvs(size=size, random_state=111)
variates_lambda_hi = dist_lambda_hi.rvs(size=size, random_state=111)


# Calculate confidence intervals using normal approximation
mean_lambda_lo = np.mean(variates_lambda_lo)
se_lambda_lo = np.sqrt(mean_lambda_lo / size)
ci_lambda_lo = np.array((
    mean_lambda_lo - 1.96 * se_lambda_lo,
    mean_lambda_lo + 1.96 * se_lambda_lo))

mean_lambda_hi = np.mean(variates_lambda_hi)
se_lambda_hi = np.sqrt(mean_lambda_hi / size)
ci_lambda_hi = np.array((
    mean_lambda_hi - 1.96 * se_lambda_hi,
    mean_lambda_hi + 1.96 * se_lambda_hi))

print(r"95% confidence interval (normal approximation and sample mean):")
print(f" sample from λ={lambda_lo}: {ci_lambda_lo.round(2)}")
print(f" sample from λ={lambda_hi}: {ci_lambda_hi.round(1)}")

# Theoretical prediction intervals (Normal approximation)
pi_lo_norm = np.array((
    lambda_lo - 1.96 * np.sqrt(lambda_lo),
    lambda_lo + 1.96 * np.sqrt(lambda_lo)), dtype=int)
pi_hi_norm = np.array((
    lambda_hi - 1.96 * np.sqrt(lambda_hi),
    lambda_hi + 1.96 * np.sqrt(lambda_hi)), dtype=int)
print()
print(r"95% theoretical prediction interval (normal approximation with λ):")
print(f" λ={lambda_lo}:", pi_lo_norm)
print(f" λ={lambda_hi}:", pi_hi_norm)

# Calculate prediction intervals
pred_int_lambda_lo = dist_lambda_lo.interval(0.95)
pred_int_lambda_hi = dist_lambda_hi.interval(0.95)
print()
print(r"95% prediction interval (equal areas around median):")
print(f" λ={lambda_lo}:", pred_int_lambda_lo)
print(f" λ={lambda_hi}:", pred_int_lambda_hi)

In [ ]:
print(r"95% confidence interval (normal approximation and sample mean):")
print(f" λ={lambda_lo}:", ci_lambda_lo.round(2))
print(
    f" λ={lambda_hi} (normalized to events/minute):",
    (ci_lambda_hi/10).round(2))
print()
print(r"95% theoretical prediction interval (normal approximation with λ):")
print(f" λ={lambda_lo}:", pi_lo_norm)
print(
    f" λ={lambda_hi} (normalized to events/minute, rounded):",
    pi_hi_norm/10)
print()
print("95% prediction interval (equal areas around median):")
print(f" λ={lambda_lo}:", pred_int_lambda_lo)
print(
    f" λ={lambda_hi} (normalized to events/minute):",
    tuple(x/10 for x in pred_int_lambda_hi))

In [ ]:

# Create boxplots (normalized for comparison)
plt.figure(figsize=(3.5, 3))
sns.boxplot(
    data=[
        variates_lambda_lo,
        variates_lambda_hi / 10],
    palette="Set2")
plt.xticks(
    [0, 1],
    [f"λ={lambda_lo}/min", f"λ={lambda_hi}/10 min"])
plt.title("Simulated Poisson counts")
plt.ylabel("Counts per minute (normalized)")
plt.xlabel("Scenario")
plt.grid(axis='y', linestyle='--')
sns.despine();

### Handling zero and very low event counts


In [ ]:
# Set the parameter for two Poisson distributions
lambdas_low = [0, 0.1]

for lambda_val in lambdas_low:
    # Calculate the *prediction interval* using poisson.interval()
    pred_int_zero = np.array(poisson.interval(0.95, lambda_val), dtype=int)

    print(f"λ={lambda_val}:")
    print(
        r" 95% prediction interval (equal areas around median):",
        pred_int_zero)

    # We calculate a CI using statsmodels.stats.rates.confint_poisson
    # with the 'exact-c' method, which is suitable for zero counts
    ci_exact = smr.confint_poisson(
        count=lambda_val,
        exposure=1,
        method='exact-c')

    print(f"Observed count={lambda_val}; exposure time=1")
    print(
        r" 95% confidence interval for λ (exact method):",
        (tuple(round(b, 3) for b in ci_exact)))
    print()


In [ ]:

# Create the figure and axes for side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(6, 3), sharey=True, sharex=True)

for i, lambda_val in enumerate(lambdas_low):
    # Generate x values
    x = np.arange(0, 5)  # Max x-value of 4 is sufficient for both low λ

    # Calculate the PMF
    pmf = poisson.pmf(x, lambda_val)

    # Create the lollipop plot on the appropriate subplot
    axes[i].stem(x, pmf, markerfmt="C3s", linefmt="C7-", basefmt="C2-")
    axes[i].set_title(f"Poisson PMF (λ = {lambda_val})")
    axes[i].set_xlabel("Number of events (k)")
    axes[i].set_ylabel("Probability")
    axes[i].set_xticks(x) #Show all the integer on x axis

# Adjust layout to prevent overlapping titles
plt.tight_layout();

## Conclusion
